In [22]:
!pip install scikit-learn pandas numpy -q

In [36]:
# Import necessary libraries
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
# from tensorflow.keras.layers import Input, Lambda, Dense, Flatten, Conv2D
from tensorflow.keras.models import load_model
from tensorflow.keras.applications import VGG16, ResNet50V2, EfficientNetB0, DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import os
import cv2
from datetime import datetime
import shutil 

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)


In [24]:
def load_trained_model(model_path, custom_objects=None):
    """
    Load a trained Keras model

    Parameters:
    - model_path: path to the .h5 model file
    - custom_objects: dictionary of custom layers/functions (if any)

    Returns:
    - loaded model
    """
    try:
        if custom_objects:
            model = load_model(model_path, custom_objects=custom_objects)
        else:
            model = load_model(model_path)

        print(f" Model loaded successfully from: {model_path}")
        print(f"Model input shape: {model.input_shape}")
        print(f"Model output shape: {model.output_shape}")
        return model

    except Exception as e:
        print(f" Error loading model: {str(e)}")
        return None

In [27]:
model_path = os.path.join('5', 'ResNet50V2_final_model.h5')
model = load_trained_model(model_path)
# model.summary()

 Model loaded successfully from: 5\ResNet50V2_final_model.h5
Model input shape: (None, 224, 224, 3)
Model output shape: (None, 2)


In [26]:
def predict_image(model, image_path, class_names, img_size=(224,224)):
    """
    Function to predict class and probability for a single image

    Parameters:
    - model: trained Keras model
    - image_path: path to the image file
    - class_names: list of class names
    - img_size: tuple of (height, width) for resizing

    Returns:
    - predicted_class: string, name of predicted class
    - probabilities: dictionary with class names and their probabilities
    """
    # Load and preprocess image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, img_size)
    image = image.astype('float32') / 255.0
    image = np.expand_dims(image, axis=0)

    # Make prediction
    predictions = model.predict(image, verbose=0)[0]

    # Get predicted class
    predicted_idx = np.argmax(predictions)
    predicted_class = class_names[predicted_idx]

    # Create probabilities dictionary
    probabilities = {class_names[i]: float(predictions[i]) for i in range(len(class_names))}

    return predicted_class, probabilities


In [38]:

for image_path in glob.glob(os.path.join('Dataset', 'Brain Tumor Image DataSet  Semantic Segmentation', '*', '*.jpg')):
    class_names = ['Meduloblastoma', 'Normal']
    result, proba = predict_image(model, image_path, class_names, img_size=(224,224))

    if result == 'Meduloblastoma':
        print(f"{image_path.split('\\')[-2]} | {image_path} | {result}")
        # Create the folder safely
        save_dir = image_path.split('\\')[-2]
        os.makedirs(save_dir, exist_ok=True)

        source = image_path
        destination = os.path.join(save_dir, image_path.split('\\')[-1])

        dest = shutil.copy(source, destination) 
    
    

test | Dataset\Brain Tumor Image DataSet  Semantic Segmentation\test\1030_jpg.rf.4bfc0ebc72d835cb9668219cd911c249.jpg | Meduloblastoma
test | Dataset\Brain Tumor Image DataSet  Semantic Segmentation\test\1035_jpg.rf.0dead1ccbe6b1e137ca2d98fc4d34b9a.jpg | Meduloblastoma
test | Dataset\Brain Tumor Image DataSet  Semantic Segmentation\test\1056_jpg.rf.f539615badbea313d68c957e76642dda.jpg | Meduloblastoma
test | Dataset\Brain Tumor Image DataSet  Semantic Segmentation\test\1058_jpg.rf.f6d6dcde9564c091b30eb6873641ba36.jpg | Meduloblastoma
test | Dataset\Brain Tumor Image DataSet  Semantic Segmentation\test\1083_jpg.rf.b1473a76f8f3789c6be693dfbe9bc011.jpg | Meduloblastoma
test | Dataset\Brain Tumor Image DataSet  Semantic Segmentation\test\1093_jpg.rf.4cde54a1271fe5ecc1f9d6ca4b00f835.jpg | Meduloblastoma
test | Dataset\Brain Tumor Image DataSet  Semantic Segmentation\test\1125_jpg.rf.6a2f54deeac86ff2d961132a828c9ae6.jpg | Meduloblastoma
test | Dataset\Brain Tumor Image DataSet  Semantic Segm

In [40]:
import json
import os

def filter_coco_by_images(coco_json_path, images_folder, output_json_path=None):
    """
    Filter a COCO JSON to only include annotations for images present in a folder.

    Args:
        coco_json_path:   Path to the original COCO JSON file.
        images_folder:    Path to the folder containing images.
        output_json_path: If provided, saves the filtered JSON to this path.

    Returns:
        Filtered COCO dict.
    """
    with open(coco_json_path, 'r') as f:
        coco = json.load(f)

    # Get all filenames present in the folder
    valid_filenames = set(os.listdir(images_folder))
    print(f"Images in folder : {len(valid_filenames)}")
    print(f"Images in JSON   : {len(coco['images'])}")

    # Filter images whose file_name exists in the folder
    filtered_images = [
        img for img in coco['images']
        if img['file_name'] in valid_filenames
    ]
    filtered_image_ids = {img['id'] for img in filtered_images}

    # Filter annotations belonging to those image ids
    filtered_annotations = [
        ann for ann in coco['annotations']
        if ann['image_id'] in filtered_image_ids
    ]

    # Build filtered COCO dict (keep categories and info as-is)
    filtered_coco = {
        'info':        coco.get('info', {}),
        'licenses':    coco.get('licenses', []),
        'categories':  coco.get('categories', []),
        'images':      filtered_images,
        'annotations': filtered_annotations,
    }

    print(f"Filtered images      : {len(filtered_images)}")
    print(f"Filtered annotations : {len(filtered_annotations)}")

    if output_json_path:
        with open(output_json_path, 'w') as f:
            json.dump(filtered_coco, f, indent=2)
        print(f"Saved to: {output_json_path}")

    return filtered_coco


filtered = filter_coco_by_images(
    coco_json_path   = os.path.join("Dataset", "Semantic Segmentation", "train", "_annotations.coco.json"),
    images_folder    = os.path.join("Dataset", "Semantic Segmentation", "train"),
    output_json_path = "train_filtered_annotations.json",
)

Images in folder : 1298
Images in JSON   : 1502
Filtered images      : 1297
Filtered annotations : 1296
Saved to: train_filtered_annotations.json


In [41]:
filtered = filter_coco_by_images(
    coco_json_path   = os.path.join("Dataset", "Semantic Segmentation", "test", "_annotations.coco.json"),
    images_folder    = os.path.join("Dataset", "Semantic Segmentation", "test"),
    output_json_path = "test_filtered_annotations.json",
)

Images in folder : 198
Images in JSON   : 215
Filtered images      : 197
Filtered annotations : 197
Saved to: test_filtered_annotations.json


In [42]:
filtered = filter_coco_by_images(
    coco_json_path   = os.path.join("Dataset", "Semantic Segmentation", "valid", "_annotations.coco.json"),
    images_folder    = os.path.join("Dataset", "Semantic Segmentation", "valid"),
    output_json_path = "_valid_filtered_annotations.json",
)

Images in folder : 375
Images in JSON   : 429
Filtered images      : 374
Filtered annotations : 374
Saved to: _valid_filtered_annotations.json
